# Automated RGC Detection and Cell-Death Quantification Workflow

**Goal:** Detect retinal ganglion cells (RGCs) from the green RBPMS channel, classify each detected RGC as death-marker positive or negative from the red channel, then compare:

\[
\frac{\text{death-marker-positive RGCs}}{\text{total RGCs}}
\]

between a **control** group and a **manipulation** group.

This notebook assumes:
- Images are already **max intensity projections**.
- Each image has two channels:
  - **Green** = RBPMS, total RGC marker.
  - **Red** = death/dying-cell marker.
- The red marker is treated as binary-like: bright/present vs dim/absent.
- There may be thousands to millions of images, so the workflow avoids excessive plotting.


## Library roles

- **numpy**: numerical array operations.
- **pandas**: tables for per-cell, per-image, and per-group results.
- **tifffile**: fast loading of microscopy TIFF images.
- **scikit-image**: image filtering, thresholding, morphology, watershed segmentation, region measurements.
- **scipy**: distance transform and statistics.
- **scikit-learn**: optional machine-learning classifier for death-marker positivity.
- **joblib**: parallel processing across many images.
- **matplotlib**: minimal quality-control plots only.


In [ ]:
# Core libraries
from pathlib import Path
import numpy as np
import pandas as pd

# Image I/O
import tifffile as tiff

# Image processing / segmentation
from skimage import filters, measure, morphology, exposure, segmentation
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
from scipy import stats

# Optional ML classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from joblib import Parallel, delayed

# Minimal plotting for QC only
import matplotlib.pyplot as plt


## 1. Define paths and group labels

Expected folder structure:

```text
data/
    control/
        image_001.tif
        image_002.tif
    manipulation/
        image_001.tif
        image_002.tif
```

Each image should contain both channels. Common formats are:
- `(channels, y, x)`
- `(y, x, channels)`

The helper function below tries to separate green and red automatically.


In [ ]:
DATA_DIR = Path("data")

GROUP_DIRS = {
    "control": DATA_DIR / "control",
    "manipulation": DATA_DIR / "manipulation",
}

IMAGE_EXTENSIONS = ["*.tif", "*.tiff"]

image_table = []

for group, folder in GROUP_DIRS.items():
    for ext in IMAGE_EXTENSIONS:
        for path in folder.glob(ext):
            image_table.append({"group": group, "image_path": path})

image_df = pd.DataFrame(image_table)
print(image_df["group"].value_counts())
image_df.head()


## 2. Load two-channel image

This function reads one image and returns:

- `green`: RBPMS channel used for RGC detection.
- `red`: death-marker channel used for death-positive classification.

If your channel order is different, change `green_index` and `red_index`.


In [ ]:
def load_green_red_image(path, green_index=0, red_index=1):
    img = tiff.imread(path)

    if img.ndim != 3:
        raise ValueError(f"Expected 3D image with 2 channels, got shape {img.shape}")

    # Case 1: channels first: (C, Y, X)
    if img.shape[0] <= 4:
        green = img[green_index]
        red = img[red_index]

    # Case 2: channels last: (Y, X, C)
    elif img.shape[-1] <= 4:
        green = img[..., green_index]
        red = img[..., red_index]

    else:
        raise ValueError(
            f"Could not infer channel axis from shape {img.shape}. "
            "Modify load_green_red_image manually."
        )

    return green.astype(np.float32), red.astype(np.float32)


## 3. Segment RGCs from the green RBPMS channel

This step detects total RGCs.

Workflow:
1. Normalize image intensity.
2. Smooth noise.
3. Threshold the RBPMS signal.
4. Clean the binary mask.
5. Use watershed to separate touching cells.
6. Remove objects outside expected RGC size range.

Tune `MIN_CELL_AREA` and `MAX_CELL_AREA` for your microscope magnification and image scale.


In [ ]:
MIN_CELL_AREA = 40
MAX_CELL_AREA = 1500
GAUSSIAN_SIGMA = 1.0

def segment_rgcs(green):
    # Normalize contrast
    green_norm = exposure.rescale_intensity(green, out_range=(0, 1))

    # Smooth to reduce pixel noise
    smooth = filters.gaussian(green_norm, sigma=GAUSSIAN_SIGMA)

    # Automatic threshold
    threshold = filters.threshold_otsu(smooth)
    binary = smooth > threshold

    # Morphological cleanup
    binary = morphology.remove_small_objects(binary, min_size=MIN_CELL_AREA)
    binary = morphology.remove_small_holes(binary, area_threshold=MIN_CELL_AREA)

    # Watershed separation for touching RGCs
    distance = ndi.distance_transform_edt(binary)
    coords = peak_local_max(distance, labels=binary, min_distance=5)
    markers = np.zeros(distance.shape, dtype=int)
    markers[tuple(coords.T)] = np.arange(1, len(coords) + 1)
    markers = ndi.label(markers > 0)[0]

    labels = segmentation.watershed(-distance, markers, mask=binary)

    # Filter by area
    cleaned = np.zeros_like(labels)
    new_label = 1

    for region in measure.regionprops(labels, intensity_image=green_norm):
        if MIN_CELL_AREA <= region.area <= MAX_CELL_AREA:
            cleaned[labels == region.label] = new_label
            new_label += 1

    return cleaned


## 4. Extract per-cell features

For every segmented RGC, measure:
- RGC area.
- Green/RBPMS intensity.
- Red/death-marker intensity inside the cell.
- Local red background.
- Red signal corrected for background.
- Red-to-green ratio.

These features can be used either for simple thresholding or for ML classification.


In [ ]:
def extract_cell_features(green, red, labels, image_path, group):
    rows = []

    green_norm = exposure.rescale_intensity(green, out_range=(0, 1))
    red_norm = exposure.rescale_intensity(red, out_range=(0, 1))

    regions = measure.regionprops(labels, intensity_image=green_norm)

    for region in regions:
        cell_mask = labels == region.label

        # Dilated ring around cell for local background estimate
        dilated = morphology.binary_dilation(cell_mask, morphology.disk(6))
        ring = dilated & ~cell_mask

        red_mean = red_norm[cell_mask].mean()
        red_max = red_norm[cell_mask].max()
        red_bg = red_norm[ring].median() if np.any(ring) else np.median(red_norm)

        green_mean = green_norm[cell_mask].mean()

        rows.append({
            "group": group,
            "image_path": str(image_path),
            "cell_label": region.label,
            "area": region.area,
            "centroid_y": region.centroid[0],
            "centroid_x": region.centroid[1],
            "green_mean": green_mean,
            "red_mean": red_mean,
            "red_max": red_max,
            "red_local_bg": red_bg,
            "red_bg_corrected": red_mean - red_bg,
            "red_to_green_ratio": red_mean / (green_mean + 1e-8),
        })

    return pd.DataFrame(rows)


## 5. Classify death-marker positivity

Two options are shown.

### Option A: automatic thresholding

This is useful when the red channel is strongly binary.

### Option B: supervised ML

Use this if you manually label a small subset of cells as:
- `death_positive = 1`
- `death_positive = 0`

For many biological imaging datasets, a small manually labeled training set can greatly improve reliability.


In [ ]:
def classify_death_by_threshold(cell_df):
    # Automatic threshold on background-corrected red intensity
    threshold = filters.threshold_otsu(cell_df["red_bg_corrected"].values)
    cell_df = cell_df.copy()
    cell_df["death_positive"] = cell_df["red_bg_corrected"] > threshold
    cell_df["death_classifier"] = "otsu_threshold"
    cell_df["death_threshold"] = threshold
    return cell_df


def train_death_classifier(labeled_cell_df):
    features = [
        "area",
        "green_mean",
        "red_mean",
        "red_max",
        "red_local_bg",
        "red_bg_corrected",
        "red_to_green_ratio",
    ]

    X = labeled_cell_df[features]
    y = labeled_cell_df["death_positive"].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(classification_report(y_test, preds))

    return model, features


def classify_death_by_ml(cell_df, model, features):
    cell_df = cell_df.copy()
    cell_df["death_positive"] = model.predict(cell_df[features]).astype(bool)
    cell_df["death_probability"] = model.predict_proba(cell_df[features])[:, 1]
    cell_df["death_classifier"] = "random_forest"
    return cell_df


## 6. Process one image

This combines:
1. Load image.
2. Segment RGCs from green/RBPMS.
3. Extract red-channel death features.
4. Classify death-marker positivity.


In [ ]:
def process_one_image(row, ml_model=None, ml_features=None):
    image_path = Path(row["image_path"])
    group = row["group"]

    green, red = load_green_red_image(image_path)
    labels = segment_rgcs(green)
    cell_df = extract_cell_features(green, red, labels, image_path, group)

    if len(cell_df) == 0:
        return cell_df

    if ml_model is None:
        cell_df = classify_death_by_threshold(cell_df)
    else:
        cell_df = classify_death_by_ml(cell_df, ml_model, ml_features)

    return cell_df


## 7. Minimal quality-control check on one image

This plots only one example image to confirm that segmentation is reasonable.

For large datasets, do not plot every image.


In [ ]:
# Minimal QC on one image
example = image_df.iloc[0]
green, red = load_green_red_image(example["image_path"])
labels = segment_rgcs(green)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(green, cmap="gray")
ax[0].set_title("Green/RBPMS")
ax[1].imshow(red, cmap="gray")
ax[1].set_title("Red/death marker")
ax[2].imshow(segmentation.mark_boundaries(exposure.rescale_intensity(green, out_range=(0, 1)), labels))
ax[2].set_title("Detected RGCs")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()


## 8. Batch process all images

For thousands or millions of images, use parallel processing.

Adjust `N_JOBS` based on your CPU/RAM.


In [ ]:
N_JOBS = 8

all_cell_tables = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(process_one_image)(row)
    for _, row in image_df.iterrows()
)

cell_results = pd.concat(all_cell_tables, ignore_index=True)

cell_results.to_csv("rgc_cell_level_results.csv", index=False)
cell_results.head()


## 9. Summarize per image

Each image gets:
- Total RGC count.
- Death-positive RGC count.
- Fraction death positive.


In [ ]:
image_summary = (
    cell_results
    .groupby(["group", "image_path"])
    .agg(
        total_rgcs=("cell_label", "count"),
        death_positive_rgcs=("death_positive", "sum"),
    )
    .reset_index()
)

image_summary["fraction_death_positive"] = (
    image_summary["death_positive_rgcs"] / image_summary["total_rgcs"]
)

image_summary.to_csv("rgc_image_level_summary.csv", index=False)
image_summary.head()


## 10. Summarize per group

This produces the main biological endpoint:

\[
\text{death fraction} =
\frac{\text{death-positive RGCs}}{\text{total detected RGCs}}
\]


In [ ]:
group_summary = (
    image_summary
    .groupby("group")
    .agg(
        n_images=("image_path", "count"),
        total_rgcs=("total_rgcs", "sum"),
        death_positive_rgcs=("death_positive_rgcs", "sum"),
        mean_fraction_death_positive=("fraction_death_positive", "mean"),
        sd_fraction_death_positive=("fraction_death_positive", "std"),
        sem_fraction_death_positive=("fraction_death_positive", lambda x: x.std() / np.sqrt(len(x))),
    )
    .reset_index()
)

group_summary["pooled_fraction_death_positive"] = (
    group_summary["death_positive_rgcs"] / group_summary["total_rgcs"]
)

group_summary.to_csv("rgc_group_level_summary.csv", index=False)
group_summary


## 11. Compare control vs manipulation

This compares image-level death fractions between groups.

Use image-level values rather than treating every cell as fully independent, because cells from the same image are not truly independent biological replicates.


In [ ]:
control = image_summary.loc[
    image_summary["group"] == "control",
    "fraction_death_positive"
].dropna()

manipulation = image_summary.loc[
    image_summary["group"] == "manipulation",
    "fraction_death_positive"
].dropna()

# Welch's t-test: good default when group variances/sample sizes may differ
t_stat, p_value = stats.ttest_ind(control, manipulation, equal_var=False)

comparison_results = pd.DataFrame({
    "comparison": ["manipulation_vs_control"],
    "control_mean": [control.mean()],
    "manipulation_mean": [manipulation.mean()],
    "difference": [manipulation.mean() - control.mean()],
    "welch_t_stat": [t_stat],
    "p_value": [p_value],
})

comparison_results.to_csv("rgc_group_comparison_results.csv", index=False)
comparison_results


## 12. Minimal final plot

This is the only recommended routine plot for large datasets: group-level death-positive fraction.


In [ ]:
plot_df = image_summary.copy()

plt.figure(figsize=(5, 4))
plot_df.boxplot(
    column="fraction_death_positive",
    by="group",
    grid=False
)
plt.title("Death-positive RGC fraction by group")
plt.suptitle("")
plt.xlabel("Group")
plt.ylabel("Death-positive RGCs / total RGCs")
plt.tight_layout()
plt.savefig("rgc_death_fraction_group_comparison.png", dpi=300)
plt.show()


## Notes for scaling to very large datasets

For thousands to millions of images:

1. Keep per-cell results as CSV or Parquet files split by batch.
2. Avoid plotting during batch processing.
3. Validate segmentation on a small random subset before full analysis.
4. Consider saving segmentation masks only for QC subsets, not every image.
5. Use supervised ML if red-channel positivity is ambiguous.
6. Use image-level, animal-level, or retina-level summaries for statistics, depending on the true biological replicate.
7. If RGC morphology is highly variable, replace classical watershed segmentation with a deep-learning segmentation model such as Cellpose, StarDist, or a U-Net trained on manually labeled RGC masks.
